# 21B — V5 E1 Apply Manual Overrides + Final Event Freeze

This notebook applies the **already frozen** source/semantic resolutions from 21A manual backfill.

It may:
- verify hashes,
- apply the exact 14 predeclared corrections,
- validate source coverage and axis integrity,
- freeze the E1 event corpus.

It may **not**:
- add new events,
- change roster membership,
- inspect 1–5 year pair gaps,
- generate pairs,
- rebalance chronology,
- generate astrology,
- score Control,
- research CONFIRM.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, re
import numpy as np
import pandas as pd

NOTEBOOK_VERSION="SAJU_ML_V5_E1_FINAL_EVENT_FREEZE_20260817"

def repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repo.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

ROOT=repo_root()
QA=ROOT/"research/ml/artifacts/v5_dev_expansion_e1_source_semantic_qa"
E1=ROOT/"research/ml/artifacts/v5_dev_expansion_e1"
CORPUS=ROOT/"research/ml_corpus/v5_ground_truth"
OUT=ROOT/"research/ml/artifacts/v5_dev_expansion_e1_event_freeze"
OUT.mkdir(parents=True,exist_ok=True)

ASSEMBLED=QA/"V5_E1_EVENTS_ASSEMBLED_PRE_QA.csv"
AUTO=QA/"V5_E1_SOURCE_QA_AUTO_RESULTS.csv"
SRC_REVIEW=QA/"V5_E1_ELIGIBLE_SOURCE_MANUAL_REVIEW.csv"
SEM_REVIEW=QA/"V5_E1_ELIGIBLE_SEMANTIC_MANUAL_REVIEW.csv"
COLL_REVIEW=QA/"V5_E1_SAME_YEAR_OPPOSITE_POLARITY_REVIEW.csv"
DECISION=QA/"V5_E1_SOURCE_SEMANTIC_QA_DECISION.json"
SUMMARY=QA/"V5_E1_SOURCE_SEMANTIC_QA_SUMMARY.json"

SRC_OVR=QA/"V5_E1_SOURCE_MANUAL_OVERRIDE.csv"
SEM_OVR=QA/"V5_E1_SEMANTIC_MANUAL_OVERRIDE.csv"
CORR=QA/"V5_E1_EVENT_CORRECTIONS.csv"
BACKFILL=QA/"V5_E1_MANUAL_BACKFILL_SUMMARY.json"
PROTO=CORPUS/"V5_E1_FINAL_EVENT_FREEZE_PROTOCOL.json"
ROSTER=E1/"V5_DEV_EXPANSION_E1_ROSTER_160.csv"

for p in [ASSEMBLED,AUTO,SRC_REVIEW,SEM_REVIEW,COLL_REVIEW,DECISION,SUMMARY,
          SRC_OVR,SEM_OVR,CORR,BACKFILL,PROTO,ROSTER]:
    if not p.exists(): raise FileNotFoundError(p)

dec=json.load(open(DECISION,encoding="utf-8"))
summ=json.load(open(SUMMARY,encoding="utf-8"))
back=json.load(open(BACKFILL,encoding="utf-8"))
proto=json.load(open(PROTO,encoding="utf-8"))

assert dec["status"]=="V5_E1_SOURCE_SEMANTIC_MANUAL_BACKFILL_REQUIRED"
assert proto["status"]=="PREDECLARED_AFTER_MANUAL_BACKFILL_BEFORE_E1_EVENT_FREEZE"
assert back["status"]=="V5_E1_MANUAL_SOURCE_SEMANTIC_BACKFILL_COMPLETE_READY_FOR_FINAL_FREEZE"

assert sha256_file(ASSEMBLED)==dec["assembled_events_sha256"]
assert sha256_file(AUTO)==dec["source_auto_results_sha256"]
assert sha256_file(SRC_REVIEW)==dec["eligible_source_manual_review_sha256"]
assert sha256_file(SEM_REVIEW)==dec["eligible_semantic_manual_review_sha256"]
assert sha256_file(COLL_REVIEW)==dec["same_year_collision_review_sha256"]
assert sha256_file(SUMMARY)==dec["summary_sha256"]

assert sha256_file(SRC_REVIEW)==back["source_review_input_sha256"]
assert sha256_file(SEM_REVIEW)==back["semantic_review_input_sha256"]
assert sha256_file(COLL_REVIEW)==back["same_year_review_input_sha256"]
assert sha256_file(SRC_OVR)==back["source_manual_override_sha256"]
assert sha256_file(SEM_OVR)==back["semantic_manual_override_sha256"]
assert sha256_file(CORR)==back["event_corrections_sha256"]

events=pd.read_csv(ASSEMBLED)
auto=pd.read_csv(AUTO)
src_review=pd.read_csv(SRC_REVIEW)
sem_review=pd.read_csv(SEM_REVIEW)
src_ovr=pd.read_csv(SRC_OVR)
sem_ovr=pd.read_csv(SEM_OVR)
corr=pd.read_csv(CORR)
roster=pd.read_csv(ROSTER)

assert len(events)==333
assert events.event_row_id.is_unique
assert len(src_review)==len(src_ovr)==99
assert set(src_review.event_row_id)==set(src_ovr.event_row_id)
assert len(sem_review)==len(sem_ovr)==98
assert set(sem_review.event_row_id)==set(sem_ovr.event_row_id)
assert len(corr)==14 and corr.event_row_id.is_unique
assert src_ovr.event_fact_verified.astype(bool).all()

print("21B PREFLIGHT PASS")


21B PREFLIGHT PASS


## 1. Apply the exact frozen correction set

In [2]:

x=events.copy()
x["pre_qa_event_row_id"]=x["event_row_id"]

# Preserve original values for correction audit.
for c in ["event_year","event_type","event_description","source_url","exclude","exclude_reason"]:
    x[f"pre_correction_{c}"]=x[c]

corr_idx=corr.set_index("event_row_id")

for rid,r in corr_idx.iterrows():
    mask=x.pre_qa_event_row_id==rid
    assert mask.sum()==1, rid

    if r.action=="SET_EXCLUDE":
        x.loc[mask,"exclude"]=True
        x.loc[mask,"exclude_reason"]=str(r.corrected_exclude_reason)

    elif r.action=="CORRECT_FIELDS":
        x.loc[mask,"event_year"]=int(float(r.corrected_event_year))
        x.loc[mask,"event_type"]=str(r.corrected_event_type)
        x.loc[mask,"event_description"]=str(r.corrected_event_description)
        x.loc[mask,"source_url"]=str(r.corrected_source_url)

    else:
        raise ValueError(f"Unknown correction action {r.action}")

assert len(x)==333
assert int((~x.exclude.astype(bool)).sum())==277
assert int(x.exclude.astype(bool).sum())==56

print("Correction application PASS")
print(corr.action.value_counts())


Correction application PASS
SET_EXCLUDE       10
CORRECT_FIELDS     4
Name: action, dtype: int64


## 2. Freeze final source verification state

In [3]:

auto_status=auto.set_index("event_row_id")["source_verification_status"].to_dict()
manual_status=src_ovr.set_index("event_row_id")["manual_source_status"].to_dict()
corr_source_verified=corr.set_index("event_row_id")["correction_source_verified"].astype(bool).to_dict()

def final_source_status(r):
    rid=r.pre_qa_event_row_id
    if rid in corr_source_verified and corr_source_verified[rid] and rid in set(
        corr.loc[corr.action=="CORRECT_FIELDS","event_row_id"]
    ):
        return "VERIFIED_CORRECTION_SOURCE"
    if auto_status.get(rid)=="AUTO_PASS":
        return "VERIFIED_AUTO"
    if rid in manual_status:
        return manual_status[rid]
    return auto_status.get(rid,"UNVERIFIED")

x["final_source_verification"]=x.apply(final_source_status,axis=1)

eligible=x[~x.exclude.astype(bool)].copy()
verified_values={
    "VERIFIED_AUTO",
    "VERIFIED_ORIGINAL_MANUAL",
    "VERIFIED_ALTERNATE",
    "VERIFIED_CORRECTION_SOURCE"
}
bad=eligible[~eligible.final_source_verification.isin(verified_values)]
assert len(bad)==0, bad[
    ["pre_qa_event_row_id","subject_id","event_year","source_url","final_source_verification"]
].to_dict(orient="records")[:20]

print("Eligible source verification coverage: 100%")
print(eligible.final_source_verification.value_counts())


Eligible source verification coverage: 100%
VERIFIED_AUTO                 180
VERIFIED_ORIGINAL_MANUAL       90
VERIFIED_CORRECTION_SOURCE      4
VERIFIED_ALTERNATE              3
Name: final_source_verification, dtype: int64


## 3. Semantic resolution and structural integrity

In [4]:

# All rows that 21A explicitly flagged semantically must have a resolution.
assert set(sem_ovr.semantic_resolution).issubset({"KEEP","EXCLUDE","CORRECT_FIELDS"})
assert sem_ovr.event_row_id.nunique()==98

# Every semantic EXCLUDE/CORRECT_FIELDS row must appear in the correction table.
sem_changed=set(
    sem_ovr.loc[sem_ovr.semantic_resolution.isin(["EXCLUDE","CORRECT_FIELDS"]),"event_row_id"]
)
assert sem_changed.issubset(set(corr.event_row_id))

assert roster.subject_id.nunique()==160
assert x.subject_id.nunique()==160
assert set(x.subject_id)==set(roster.subject_id)

roster_axis=roster.set_index("subject_id")["preassigned_axis"]
assert x.preassigned_axis.eq(x.subject_id.map(roster_axis)).all()

eligible=x[~x.exclude.astype(bool)].copy()
assert set(eligible.polarity.unique()).issubset({"positive","negative"})
assert eligible.event_year.notna().all()
assert eligible.event_description.fillna("").str.strip().ne("").all()
assert eligible.source_url.fillna("").str.startswith(("http://","https://")).all()

# Same-year opposite polarity is a semantic audit only; it is retained where
# distinct events legitimately occurred. This is NOT a 1-5y pair calculation.
yrpol=eligible.groupby(["subject_id","event_year"]).polarity.agg(lambda z:set(z))
same_year=yrpol[yrpol.map(lambda s:{"positive","negative"}.issubset(s))]

print("STRUCTURAL QA PASS")
print("same-year opposite-polarity subject-years retained:",len(same_year))


STRUCTURAL QA PASS
same-year opposite-polarity subject-years retained: 8


## 4. Duplicate QA and stable frozen IDs

In [5]:

dup_key=["subject_id","event_year","polarity","event_type","event_description"]
dups=eligible[eligible.duplicated(dup_key,keep=False)].sort_values(dup_key).copy()
dup_path=OUT/"V5_E1_EVENT_DUPLICATE_REVIEW_CANDIDATES.csv"
dups.to_csv(dup_path,index=False)

if len(dups):
    raise RuntimeError(
        f"Freeze blocked: {len(dups)} exact semantic duplicate candidates remain. "
        "Review without using pairability/chronology."
    )

def frozen_id(r):
    raw="|".join([
        "E1_FROZEN",
        str(r.subject_id),
        str(r.event_year),
        str(r.polarity),
        str(r.event_type),
        str(r.event_description),
        str(r.source_url),
        str(bool(r.exclude))
    ])
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

x["frozen_event_row_id"]=x.apply(frozen_id,axis=1)
assert x.frozen_event_row_id.is_unique

print("DUPLICATE QA PASS")


DUPLICATE QA PASS


## 5. Freeze E1 corpus

In [6]:

sort_cols=["subject_id","event_year","polarity","event_type","frozen_event_row_id"]
full=x.sort_values(sort_cols,kind="stable").reset_index(drop=True)
eligible=full[~full.exclude.astype(bool)].copy().reset_index(drop=True)

full_path=OUT/"V5_E1_EVENT_CORPUS_FULL_FROZEN.csv"
eligible_path=OUT/"V5_E1_EVENT_CORPUS_ELIGIBLE_FROZEN.csv"
full.to_csv(full_path,index=False)
eligible.to_csv(eligible_path,index=False)

corr_audit_cols=[
    "pre_qa_event_row_id","frozen_event_row_id","subject_id","name","preassigned_axis",
    "pre_correction_event_year","event_year",
    "pre_correction_event_type","event_type",
    "pre_correction_event_description","event_description",
    "pre_correction_source_url","source_url",
    "pre_correction_exclude","exclude",
    "pre_correction_exclude_reason","exclude_reason",
    "final_source_verification"
]
corr_ids=set(corr.event_row_id)
corr_audit=full[full.pre_qa_event_row_id.isin(corr_ids)][corr_audit_cols].copy()
corr_audit_path=OUT/"V5_E1_EVENT_CORRECTION_AUDIT.csv"
corr_audit.to_csv(corr_audit_path,index=False)
assert len(corr_audit)==14

verification=full[[
    "pre_qa_event_row_id","frozen_event_row_id","subject_id","name","event_year",
    "polarity","event_type","source_url","exclude","final_source_verification"
]].copy()
verification=verification.merge(
    src_ovr[["event_row_id","verified_source_url","manual_source_status","verification_note"]],
    left_on="pre_qa_event_row_id",right_on="event_row_id",how="left",validate="one_to_one"
).drop(columns=["event_row_id"])
verification_path=OUT/"V5_E1_EVENT_SOURCE_VERIFICATION_FINAL.csv"
verification.to_csv(verification_path,index=False)

axis_cov=(
    eligible.groupby(["preassigned_axis","polarity"])
    .size().rename("eligible_event_rows").reset_index()
)
sub_cov=(
    eligible.groupby(["subject_id","preassigned_axis"])
    .agg(
        positive_events=("polarity",lambda z:int((z=="positive").sum())),
        negative_events=("polarity",lambda z:int((z=="negative").sum())),
        distinct_event_years=("event_year","nunique")
    ).reset_index()
)
# Include zero-eligible roster subjects in subject coverage.
base=roster[["subject_id","preassigned_axis"]].drop_duplicates()
sub_cov=base.merge(sub_cov,on=["subject_id","preassigned_axis"],how="left")
for c in ["positive_events","negative_events","distinct_event_years"]:
    sub_cov[c]=sub_cov[c].fillna(0).astype(int)

axis_cov_path=OUT/"V5_E1_EVENT_FREEZE_AXIS_POLARITY_COVERAGE.csv"
sub_cov_path=OUT/"V5_E1_EVENT_FREEZE_SUBJECT_COVERAGE.csv"
axis_cov.to_csv(axis_cov_path,index=False)
sub_cov.to_csv(sub_cov_path,index=False)

manifest={
    "version":"V5_E1_EVENT_FREEZE_MANIFEST_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":"V5_E1_EVENT_CORPUS_FROZEN_READY_FOR_COMBINED_PREPAIR_GATE",
    "E1_subjects_n":160,
    "full_event_rows_n":int(len(full)),
    "eligible_event_rows_n":int(len(eligible)),
    "excluded_audit_rows_n":int(full.exclude.astype(bool).sum()),
    "eligible_source_verified_n":int(
        eligible.final_source_verification.isin({
            "VERIFIED_AUTO","VERIFIED_ORIGINAL_MANUAL","VERIFIED_ALTERNATE","VERIFIED_CORRECTION_SOURCE"
        }).sum()
    ),
    "manual_source_review_rows_resolved_n":99,
    "manual_semantic_review_rows_resolved_n":98,
    "corrections_applied_n":14,
    "set_exclude_corrections_n":10,
    "field_corrections_n":4,
    "same_year_opposite_polarity_subject_years_retained_n":int(len(same_year)),
    "full_corpus_sha256":sha256_file(full_path),
    "eligible_corpus_sha256":sha256_file(eligible_path),
    "correction_audit_sha256":sha256_file(corr_audit_path),
    "source_verification_final_sha256":sha256_file(verification_path),
    "axis_polarity_coverage_sha256":sha256_file(axis_cov_path),
    "subject_coverage_sha256":sha256_file(sub_cov_path),
    "source_override_sha256":sha256_file(SRC_OVR),
    "semantic_override_sha256":sha256_file(SEM_OVR),
    "event_corrections_sha256":sha256_file(CORR),
    "freeze_protocol_sha256":sha256_file(PROTO),
    "rules":{
        "membership_changed":False,
        "new_events_added_after_21A":False,
        "corrections_used_pairability":False,
        "pair_gap_inspected":False,
        "pairs_generated":False,
        "chronology_balanced":False,
        "astrology_generated":False,
        "control_scored":False,
        "confirm_researched":False
    },
    "combined_prepair_gate_allowed":True,
    "pair_generation_allowed":False,
    "astrology_generation_allowed":False,
    "confirm_event_research_allowed":False,
    "next_rule":(
        "Combine original frozen DEV and E1 frozen DEV subject coverage. "
        "Run the necessary-condition upper-bound gate before any 1-5y pair generation."
    )
}
manifest_path=OUT/"V5_E1_EVENT_FREEZE_MANIFEST.json"
json.dump(manifest,open(manifest_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

decision={
    "version":"V5_E1_EVENT_FREEZE_DECISION_V1",
    "status":manifest["status"],
    "full_corpus_sha256":manifest["full_corpus_sha256"],
    "eligible_corpus_sha256":manifest["eligible_corpus_sha256"],
    "manifest_sha256":sha256_file(manifest_path),
    "combined_prepair_gate_allowed":True,
    "pair_generation_allowed":False,
    "astrology_generation_allowed":False,
    "confirm_event_research_allowed":False
}
decision_path=OUT/"V5_E1_EVENT_FREEZE_DECISION.json"
json.dump(decision,open(decision_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps(manifest,ensure_ascii=False,indent=2))


{
  "version": "V5_E1_EVENT_FREEZE_MANIFEST_V1",
  "notebook_version": "SAJU_ML_V5_E1_FINAL_EVENT_FREEZE_20260817",
  "created_at": "2026-08-17T06:04:13",
  "status": "V5_E1_EVENT_CORPUS_FROZEN_READY_FOR_COMBINED_PREPAIR_GATE",
  "E1_subjects_n": 160,
  "full_event_rows_n": 333,
  "eligible_event_rows_n": 277,
  "excluded_audit_rows_n": 56,
  "eligible_source_verified_n": 277,
  "manual_source_review_rows_resolved_n": 99,
  "manual_semantic_review_rows_resolved_n": 98,
  "corrections_applied_n": 14,
  "set_exclude_corrections_n": 10,
  "field_corrections_n": 4,
  "same_year_opposite_polarity_subject_years_retained_n": 8,
  "full_corpus_sha256": "75d1bc6d9d0bb65e5039123809b8a6c9b82c838bfdfd77396582e0678983423c",
  "eligible_corpus_sha256": "a76a5b850fc3f75156765139deefca8412272098229c6e2494fca5b558c15753",
  "correction_audit_sha256": "624762bcd5ab7933454128229b202e68781e83b4010a51abc36e8dc1c0bb8584",
  "source_verification_final_sha256": "4758fc65e44cc466ce9ac5188363d70b8c95b5dc4d27963

## Return to ChatGPT

Expected success:

`V5_E1_EVENT_CORPUS_FROZEN_READY_FOR_COMBINED_PREPAIR_GATE`

Send:

```text
V5_E1_EVENT_FREEZE_DECISION.json
V5_E1_EVENT_FREEZE_MANIFEST.json
V5_E1_EVENT_FREEZE_AXIS_POLARITY_COVERAGE.csv
V5_E1_EVENT_FREEZE_SUBJECT_COVERAGE.csv
```

Do not generate pairs or astrology yet.

Next: combine Original DEV + E1 coverage and run the necessary-condition gate. Only if that passes do we finally inspect 1–5 year local pair gaps.
